# Hospital 数据集 Governance 实验

本 notebook 用于运行第二篇 governance 框架在 Hospital 数据集上的实验。

In [1]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

print("当前项目目录：", PROJECT_ROOT)


当前项目目录： e:\yan\组\三支决策\机器学习\BT_TWD_v2


In [2]:
CONFIG_PATH = "configs/hospital_bttwd.yaml"

# 是否运行消融实验：
# True：运行 no_cp 和 no_progressive 两个消融；False：只运行 full，跳过消融单元。
RUN_ABLATION = False

print("当前配置文件：", CONFIG_PATH)
print("运行消融实验：", RUN_ABLATION)


当前配置文件： configs/hospital_bttwd.yaml
运行消融实验： False


## 结果摘要工具函数


In [3]:
import pandas as pd
from pathlib import Path
from IPython.display import display

output_root = Path("outputs/governance")

def show_governance_summary(mode: str) -> None:
    """Read and display the current dataset summary for one run mode."""
    config_stem = Path(CONFIG_PATH).stem
    print(f"\n===== {mode} results =====")

    dataset_summary = output_root / mode / "dataset_summary.csv"
    fold_summary = output_root / mode / config_stem / "fold_summary.csv"
    sample_records = output_root / mode / config_stem / "sample_records.csv"

    if dataset_summary.exists():
        print("Reading:", dataset_summary)
        summary_df = pd.read_csv(dataset_summary)
        if "dataset_name" in summary_df.columns:
            dataset_row = summary_df[summary_df["dataset_name"].astype(str) == config_stem]
            if dataset_row.empty:
                available = ", ".join(summary_df["dataset_name"].astype(str).tolist())
                print(f"No row for {config_stem}; available datasets: {available}")
            else:
                display(dataset_row.reset_index(drop=True))
        else:
            print("dataset_summary.csv has no dataset_name column; cannot filter by dataset.")
    else:
        print("Missing dataset_summary.csv:", dataset_summary)

    if fold_summary.exists():
        print("Reading:", fold_summary)
        display(pd.read_csv(fold_summary))
    else:
        print("Missing fold_summary.csv:", fold_summary)

    if sample_records.exists():
        print("Sample records:", sample_records)
    else:
        print("Missing sample_records.csv:", sample_records)


## 运行完整 governance 实验


In [4]:
import subprocess

cmd = [
    sys.executable,
    "scripts/run_governance_experiments.py",
    "--config",
    CONFIG_PATH,
]

print("运行命令：", " ".join(cmd))
result = subprocess.run(cmd, text=True, capture_output=True)

print("标准输出：")
print(result.stdout)

print("错误输出：")
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(f"实验运行失败，返回码：{result.returncode}")

show_governance_summary("full")


运行命令： d:\Anaconda3\python.exe scripts/run_governance_experiments.py --config configs/hospital_bttwd.yaml
标准输出：
【INFO】【2026-05-18 19:44:53】【数据加载】文本表格 E:\yan\组\三支决策\机器学习\BT_TWD_v2\data\hospital\hospital_readmissions.csv 已读取，样本数=25000，列数=17
【INFO】【2026-05-18 19:44:53】【数据加载】标签列 readmitted 已处理完成：dropna_target=False, 丢弃样本=0, 最终样本数=25000, 正类比例=47.02%
【INFO】【2026-05-18 19:44:53】【数据集信息】名称=hospital_readmissions，样本数=25000，目标列=readmitted，正类比例=47.02%
【INFO】【2026-05-18 19:44:53】【预处理】连续特征=7个，类别特征=9个
【INFO】【2026-05-18 19:44:53】【预处理】编码后维度=45
【INFO】【2026-05-18 19:44:53】【governance】hospital_bttwd 第 1/5 折
【INFO】【2026-05-18 19:44:53】【桶树】已为样本生成桶ID，共 45 个组合
【INFO】【2026-05-18 19:44:53】【桶树】已为样本生成桶ID，共 45 个组合
【INFO】【2026-05-18 19:45:04】【governance】weak bucket resolver：strong=26，weak=38
【INFO】【2026-05-18 19:45:04】【桶树】已为样本生成桶ID，共 45 个组合
【INFO】【2026-05-18 19:45:14】【governance】hospital_bttwd 第 2/5 折
【INFO】【2026-05-18 19:45:15】【桶树】已为样本生成桶ID，共 45 个组合
【INFO】【2026-05-18 19:45:15】【桶树】已为样本生成桶ID，共 45 个组合
【INFO】【2026-05-18

,dataset_name,final_regret,final_BAC,final_F1,final_accuracy,original_regret,original_BAC,original_F1,original_accuracy,closure_rate,...,root_bnd_from_rescue_failed_recall,root_bnd_from_rescue_failed_specificity,root_bnd_from_rescue_failed_fpr,root_bnd_from_rescue_failed_fnr,root_bnd_from_rescue_failed_pred_positive_rate,root_bnd_from_rescue_failed_true_positive_rate,root_bnd_from_rescue_failed_positive_rate_gap,root_bnd_from_rescue_failed_recall_specificity_gap,root_bnd_from_rescue_failed_fp_regret_sum,root_bnd_from_rescue_failed_fn_regret_sum
0,hospital_bttwd,0.53612,0.53197,0.645318,0.50676,0.52336,0.542621,0.643999,0.51988,1.0,...,0.577982,0.396135,0.603865,0.422018,0.594937,0.344937,0.25,0.181846,125.0,138.0


Reading: outputs\governance\full\hospital_bttwd\fold_summary.csv


,dataset_name,fold_id,final_regret,final_BAC,final_F1,final_accuracy,original_regret,original_BAC,original_F1,original_accuracy,...,root_bnd_from_rescue_failed_recall,root_bnd_from_rescue_failed_specificity,root_bnd_from_rescue_failed_fpr,root_bnd_from_rescue_failed_fnr,root_bnd_from_rescue_failed_pred_positive_rate,root_bnd_from_rescue_failed_true_positive_rate,root_bnd_from_rescue_failed_positive_rate_gap,root_bnd_from_rescue_failed_recall_specificity_gap,root_bnd_from_rescue_failed_fp_regret_sum,root_bnd_from_rescue_failed_fn_regret_sum
0,hospital_bttwd,1,0.5444,0.528454,0.642384,0.5032,0.5252,0.526315,0.642520,0.5006,...,0.714286,0.320000,0.680000,0.285714,0.687500,0.218750,0.468750,0.394286,17.0,6.0
1,hospital_bttwd,2,0.5310,0.533872,0.647033,0.5086,0.5244,0.542219,0.643409,0.5196,...,0.652174,0.351351,0.648649,0.347826,0.650000,0.383333,0.266667,0.300823,24.0,24.0
2,hospital_bttwd,3,0.5352,0.533444,0.645923,0.5084,0.5238,0.547422,0.642749,0.5260,...,0.666667,0.378788,0.621212,0.333333,0.636364,0.333333,0.303030,0.287879,41.0,33.0
3,hospital_bttwd,4,0.5470,0.528318,0.641858,0.5034,0.5268,0.543769,0.642119,0.5218,...,0.500000,0.439024,0.560976,0.500000,0.540984,0.327869,0.213115,0.060976,23.0,30.0
4,hospital_bttwd,5,0.5230,0.535764,0.649392,0.5102,0.5166,0.553379,0.649199,0.5314,...,0.423077,0.473684,0.526316,0.576923,0.484375,0.406250,0.078125,-0.050607,20.0,45.0


Sample records: outputs\governance\full\hospital_bttwd\sample_records.csv


## 运行无 CP 消融


In [5]:
import subprocess

if RUN_ABLATION:
    cmd = [
        sys.executable,
        "scripts/run_governance_experiments.py",
        "--config",
        CONFIG_PATH,
        "--disable-cp-validation",
    ]

    print("运行命令：", " ".join(cmd))
    result = subprocess.run(cmd, text=True, capture_output=True)

    print("标准输出：")
    print(result.stdout)

    print("错误输出：")
    print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError(f"无 CP 消融运行失败，返回码：{result.returncode}")

    show_governance_summary("no_cp")
else:
    print("RUN_ABLATION=False，跳过无 CP 消融。")


RUN_ABLATION=False，跳过无 CP 消融。


## 运行无渐进更新消融


In [6]:
import subprocess

if RUN_ABLATION:
    cmd = [
        sys.executable,
        "scripts/run_governance_experiments.py",
        "--config",
        CONFIG_PATH,
        "--disable-progressive-update",
    ]

    print("运行命令：", " ".join(cmd))
    result = subprocess.run(cmd, text=True, capture_output=True)

    print("标准输出：")
    print(result.stdout)

    print("错误输出：")
    print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError(f"无渐进更新消融运行失败，返回码：{result.returncode}")

    show_governance_summary("no_progressive")
else:
    print("RUN_ABLATION=False，跳过无渐进更新消融。")


RUN_ABLATION=False，跳过无渐进更新消融。
